In [1]:
import numpy as np
import pandas as pd

In [31]:
def Muskingum(Mk, Mx, n, DeltaT, Qin, Qout1, ntime):
    C0 = (0.5 * DeltaT - Mk * Mx) / (0.5 * DeltaT + Mk - Mk * Mx)
    C1 = (0.5 * DeltaT + Mk * Mx) / (0.5 * DeltaT + Mk - Mk * Mx)
    C2 = 1 - C0 - C1
    Qout = np.zeros(ntime)
    Qout[0] = Qout1

    for j in range(n):
        # 逐河段演算
        for itime in range(1, ntime):
            Qout[itime] = C0 * Qin[itime] + C1 * Qin[itime - 1] + C2 * Qout[itime - 1]
        if j < n - 1:
            Qin = Qout.copy()  # 将当前河段的输出作为下一个河段的输入
    if n <= 0:
        Qout = Qin.copy()  # 如果没有河段，输出等于输入
    return Qout

def Muskingumnew2(nsub, ntime, DT, REACD, SECTN, REASCD, REAECD, RVMSKN, RVMSKXE, RVMSKKE, Qinp1, Qoutini):
    DeltaT=DT/3600.0
    Qsum = np.zeros([ntime, nsub + 1])
    Qtot = np.zeros([ntime, nsub + 1])
    Qin = np.zeros(ntime)

    # 首先将区间产流量加在河流的下断面
    for isub in range(nsub):
        for itime in range(ntime):
            Qsum[itime, REAECD[isub] - 1] = Qsum[itime, REAECD[isub] - 1] + Qinp1[itime, REASCD[isub] - 1]

    # 所有单元流域循环进行汇流演算
    for isub in range(nsub):
        # 计算当前单元的 Muskingum 参数
        Mk = RVMSKKE[REASCD[isub] - 1]
        Mx = RVMSKXE[REASCD[isub] - 1]
        n  = RVMSKN[REASCD[isub] - 1]
        Qout1 = Qoutini[REASCD[isub] - 1]

        for itime in range(ntime):
            # 上断面总流量=上游来水演算结果
            Qin[itime] = Qsum[itime, REASCD[isub] - 1]
        # 进行 Muskingum 演算
        Qout = Muskingum(Mk, Mx, n, DeltaT, Qin, Qout1, ntime)
        for itime in range(ntime):
            # 将当前单元的演算结果加在下游河段的上断面
            Qsum[itime, REAECD[isub] - 1] = Qsum[itime, REAECD[isub] - 1] + Qout[itime]
            Qtot[itime, REASCD[isub] - 1] = Qin[itime]
    for itime in range(ntime):
        Qtot[itime, nsub] = Qsum[itime, nsub]
    return Qtot

In [44]:
config = pd.read_csv("configuration.txt", sep=" ", header=0)
nsub  = config["单元流域个数"].values[0]
ntime = config["数据系列长度"].values[0]
DT    = config["模拟时段长"].values[0]

msk_params = pd.read_csv("muskingumpara.txt", sep="\t", header=0)
REACD   = msk_params["！！REACD河段编号"].values
SECTN   = msk_params["SECTN河段断面数"].values
REASCD  = msk_params["REASCD河段起点编号"].values
REAECD  = msk_params["REAECD河段终点编号"].values
RVMSKN  = msk_params["RVMSKN河段马法分段数"].values
RVMSKXE = msk_params["RVMSKXE河段马法参数XE"].values
RVMSKKE = msk_params["RVMSKKE河段马法参数KE"].values

msk_input_data = pd.read_csv("muskinguminputdata.txt", sep="\t", header=0)
Qinp1 = msk_input_data.values

msk_ini = pd.read_csv("muskinguminitial.txt", sep="\t", header=0)
Qoutini = msk_ini.values[0]

Qtot = Muskingumnew2(nsub, ntime, DT, REACD, SECTN, REASCD, REAECD, RVMSKN, RVMSKXE, RVMSKKE, Qinp1, Qoutini)